In [8]:
# @title Master Pipeline: Data Extraction (LEAKAGE FIXED - 4h Horizon)
import pandas as pd
import numpy as np
from google.colab import auth
from google.cloud import bigquery
from tqdm import tqdm

# 1. SETUP
auth.authenticate_user()
PROJECT_ID = "sepsis-prediction-2025"
client = bigquery.Client(project=PROJECT_ID)

# -------------------------------------------------------------------------
# 2. SQL QUERY (SEPSIS-3 REVISED + 4-HOUR PREDICTION GAP)
# -------------------------------------------------------------------------
print("Running BigQuery Extraction (Prediction Horizon: 4 Hours)...")

query = """
-- PART A: COHORT DEFINITION (SEPSIS-3 REVISED)
WITH abx AS (
    SELECT subject_id, stay_id, starttime AS abx_time
    FROM `physionet-data.mimiciv_3_1_icu.inputevents`
    WHERE itemid IN (225879, 225798, 225834, 225843, 225859, 225893, 225828)
),
cultures AS (
    SELECT subject_id, charttime AS culture_time
    FROM `physionet-data.mimiciv_3_1_hosp.microbiologyevents`
    WHERE spec_type_desc IS NOT NULL
),
infection_times AS (
    SELECT a.subject_id, a.stay_id, a.abx_time AS suspected_time
    FROM abx a JOIN cultures c ON a.subject_id = c.subject_id
    WHERE c.culture_time BETWEEN DATETIME_SUB(a.abx_time, INTERVAL 24 HOUR) AND DATETIME_ADD(a.abx_time, INTERVAL 24 HOUR)
),
organ_failure AS (
    -- Cardio (Low BP)
    SELECT stay_id, charttime FROM `physionet-data.mimiciv_3_1_icu.chartevents` WHERE itemid = 220052 AND valuenum < 65
    UNION ALL
    -- Metabolic (High Lactate)
    SELECT ie.stay_id, le.charttime FROM `physionet-data.mimiciv_3_1_hosp.labevents` le
    JOIN `physionet-data.mimiciv_3_1_icu.icustays` ie ON le.subject_id = ie.subject_id
    WHERE le.itemid IN (50813, 52442) AND le.valuenum > 2.0
    UNION ALL
    -- Renal (High Creatinine)
    SELECT ie.stay_id, le.charttime FROM `physionet-data.mimiciv_3_1_hosp.labevents` le
    JOIN `physionet-data.mimiciv_3_1_icu.icustays` ie ON le.subject_id = ie.subject_id
    WHERE le.itemid IN (50912, 52546) AND le.valuenum > 1.2
),
sepsis_cohort AS (
    SELECT i.stay_id, i.subject_id, MIN(o.charttime) AS onset_time, 1 AS label
    FROM infection_times i JOIN organ_failure o ON i.stay_id = o.stay_id
    AND o.charttime BETWEEN DATETIME_SUB(i.suspected_time, INTERVAL 6 HOUR) AND DATETIME_ADD(i.suspected_time, INTERVAL 24 HOUR)
    GROUP BY 1, 2
    LIMIT 20000
),
control_cohort AS (
    SELECT stay_id, subject_id, DATETIME_ADD(intime, INTERVAL 24 HOUR) as onset_time, 0 as label
    FROM `physionet-data.mimiciv_3_1_icu.icustays`
    WHERE stay_id NOT IN (SELECT stay_id FROM sepsis_cohort)
    LIMIT 20000
),
final_cohort AS (SELECT * FROM sepsis_cohort UNION ALL SELECT * FROM control_cohort),

-- PART B: EXTRACT FEATURES (WITH 4-HOUR GAP)
-- We fetch data relative to Onset, but stop 4 hours before it happens.
vitals AS (
    SELECT ie.stay_id, ce.charttime, ce.itemid, ce.valuenum
    FROM `physionet-data.mimiciv_3_1_icu.chartevents` ce
    JOIN final_cohort ie ON ce.stay_id = ie.stay_id
    WHERE ce.itemid IN (220045, 220179, 220050, 220277, 220210, 223761)
    -- CRITICAL FIX: Window is [Onset-28h] to [Onset-4h]
    AND ce.charttime BETWEEN DATETIME_SUB(ie.onset_time, INTERVAL 28 HOUR)
                         AND DATETIME_SUB(ie.onset_time, INTERVAL 4 HOUR)
),
labs AS (
    SELECT ie.stay_id, le.charttime, le.itemid, le.valuenum
    FROM `physionet-data.mimiciv_3_1_hosp.labevents` le
    JOIN final_cohort ie ON le.subject_id = ie.subject_id
    WHERE le.itemid IN (50813, 51301, 50912, 50931, 51003, 50956, 50885, 51265, 50882, 51006)
    -- CRITICAL FIX: Window is [Onset-28h] to [Onset-4h]
    AND le.charttime BETWEEN DATETIME_SUB(ie.onset_time, INTERVAL 28 HOUR)
                         AND DATETIME_SUB(ie.onset_time, INTERVAL 4 HOUR)
),
static AS (
    SELECT ie.stay_id, p.anchor_age, CASE WHEN p.gender = 'F' THEN 0 ELSE 1 END as is_male, ie.label
    FROM final_cohort ie
    JOIN `physionet-data.mimiciv_3_1_icu.icustays` icu ON ie.stay_id = icu.stay_id
    JOIN `physionet-data.mimiciv_3_1_hosp.patients` p ON icu.subject_id = p.subject_id
)

SELECT
    v.stay_id, v.charttime, v.itemid, v.valuenum, s.label, s.anchor_age, s.is_male
FROM vitals v JOIN static s ON v.stay_id = s.stay_id
UNION ALL
SELECT
    l.stay_id, l.charttime, l.itemid, l.valuenum, s.label, s.anchor_age, s.is_male
FROM labs l JOIN static s ON l.stay_id = s.stay_id
"""

try:
    df_raw = client.query(query).to_dataframe()
    print(f"Extraction Complete. Rows: {len(df_raw)}")

    if not df_raw.empty:
        # ----------------------------------------------------------------
        # 3. PROCESSING: INPUT FUSION & RESHAPING
        # ----------------------------------------------------------------
        print("Processing Time-Aware Tensors...")

        # Map IDs
        ID_MAP = {
            220045: 'HR', 220179: 'SBP', 220050: 'SBP', 220277: 'O2Sat', 220210: 'RR', 223761: 'Temp',
            50813: 'Lactate', 51301: 'WBC', 50912: 'Creatinine', 50931: 'Glucose', 51003: 'Troponin',
            50956: 'Lipase', 50885: 'Bilirubin', 51265: 'Platelets', 50882: 'Bicarbonate', 51006: 'BUN'
        }
        FEATURES = ['HR', 'SBP', 'O2Sat', 'RR', 'Temp', 'Lactate', 'WBC', 'Creatinine', 'Glucose', 'Troponin', 'Lipase', 'Bilirubin', 'Platelets', 'Bicarbonate', 'BUN']

        df_raw['feature'] = df_raw['itemid'].map(ID_MAP)
        df_raw = df_raw.dropna(subset=['feature'])

        # Fix Temp (F -> C)
        mask_temp = df_raw['feature'] == 'Temp'
        df_raw.loc[mask_temp, 'valuenum'] = (df_raw.loc[mask_temp, 'valuenum'] - 32) * 5/9

        def process_patient(group):
            # Sort
            group = group.sort_values('charttime')

            # Bin by Hour (Simple Binning 0-23)
            # Since we already filtered to a 24h window in SQL, we can just cut into 24 bins
            group['hour_idx'] = pd.cut(group['charttime'], bins=24, labels=False)

            # Pivot (Values)
            pivoted = group.pivot_table(index='hour_idx', columns='feature', values='valuenum', aggfunc='mean')
            pivoted = pivoted.reindex(columns=FEATURES)
            pivoted = pivoted.reindex(range(24))

            # MASKS (1=Present, 0=Missing)
            masks = (~pivoted.isna()).astype(float).values

            # VALUES (Forward -> Zero)
            values = pivoted.ffill().fillna(0).values

            # STATIC
            age = group['anchor_age'].max()
            gender = group['is_male'].max()
            static = np.tile([age, gender], (24, 1))

            # FUSE: [Values(15) | Masks(15) | Static(2)] = 32 Features
            fused = np.hstack([values, masks, static]).astype(np.float32)
            return fused, group['label'].max()

        X_list, y_list = [], []
        for sid, group in tqdm(df_raw.groupby('stay_id')):
            try:
                tensor, label = process_patient(group)
                X_list.append(tensor)
                y_list.append(label)
            except: continue

        X_final = np.stack(X_list)
        y_final = np.array(y_list)

        print(f"\nFINAL TENSOR SHAPE: {X_final.shape}")
        print(f"LABELS SHAPE: {y_final.shape}")
        print(f"Class Balance: {sum(y_final)/len(y_final):.2%} Sepsis")

except Exception as e:
    print(f"Error: {e}")

Running BigQuery Extraction (Prediction Horizon: 4 Hours)...
Extraction Complete. Rows: 2703213
Processing Time-Aware Tensors...


100%|██████████| 31287/31287 [04:10<00:00, 125.05it/s]



FINAL TENSOR SHAPE: (31287, 24, 32)
LABELS SHAPE: (31287,)
Class Balance: 39.40% Sepsis


In [11]:
# @title STEP 4 (FIXED): STRICT SUBJECT-LEVEL SPLIT TRAINING
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score
import copy
import numpy as np
import os

print("--- STARTING STRICT TRAINING (NO SUBJECT LEAKAGE) ---")

# 1. Get Subject IDs for Grouping
# We need to query MIMIC to map stay_id -> subject_id for our X_final list
# (Assuming df_raw from the extraction step is still available)
if 'df_raw' in locals():
    # Create a map of stay_id -> subject_id
    # We iterate through X_final's implicit order.
    # NOTE: X_final was built by iterating df_raw.groupby('stay_id').
    # So we must replicate that order to get the correct groups.

    subject_ids = []
    # Re-generating the group keys to match X_final order
    for sid, group in df_raw.groupby('stay_id'):
        # We just need the subject_id for this group
        # Check if valid shape (must match the filtering logic in Step 3)
        # To be safe, we assume X_final corresponds exactly to the successful groups
        # We'll rebuild the subject list carefully.
        pass

    # SAFER APPROACH:
    # Re-extract subject_ids using the exact same loop logic used to build X_final
    print("Re-aligning Subject IDs for Splitting...")
    ordered_subjects = []
    valid_indices = []

    idx = 0
    for sid, group in df_raw.groupby('stay_id'):
        # We simulate the try/except logic from the processor
        try:
            # Just check if it WOULD generate a valid tensor (length check)
            # (This assumes the logic matches process_patient exactly)
            # Ideally, we should have saved subject_ids during Step 3.
            # Since we didn't, we will rely on the fact that tqdm iterates sorted groups usually.

            # Quick logic check: did this stay produce 24h data?
            # If yes, append subject_id.
            # This is tricky to perfectly align without re-running.
            # LET'S DO A PROXY SPLIT:
            # We will rely on GroupShuffleSplit using stay_id if subject_id isn't handy,
            # but for MIMIC, subject_id is the strict way.

            # Let's assume df_raw has 'subject_id'.
            subj = group['subject_id'].iloc[0] # Grab subject_id
            ordered_subjects.append(subj)
        except:
            pass

    # Ensure lengths match (Critical Safety Check)
    # If they don't match, we fall back to random split to avoid crash
    if len(ordered_subjects) != len(X_final):
        print(f"⚠️ Warning: Subject list length ({len(ordered_subjects)}) != Tensor length ({len(X_final)}).")
        print("   Falling back to standard shuffle split (small leakage risk).")
        groups = None
    else:
        print("✅ Subject Alignment Successful. Using GroupShuffleSplit.")
        groups = ordered_subjects

else:
    print("⚠️ Raw dataframe missing. Falling back to standard split.")
    groups = None

# 2. Perform the Split
if groups is not None:
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    train_idx, test_idx = next(gss.split(X_final, y_final, groups=groups))

    X_train, X_test = X_final[train_idx], X_final[test_idx]
    y_train, y_test = y_final[train_idx], y_final[test_idx]
else:
    from sklearn.model_selection import train_test_split
    X_train, X_test, y_train, y_test = train_test_split(
        X_final, y_final, test_size=0.2, random_state=42, stratify=y_final
    )

print(f"Training Set: {len(y_train)} patients")
print(f"Testing Set:  {len(y_test)} patients")

# 3. Data Loaders
train_loader = DataLoader(TensorDataset(torch.FloatTensor(X_train), torch.FloatTensor(y_train)), shuffle=True, batch_size=128)
test_loader = DataLoader(TensorDataset(torch.FloatTensor(X_test), torch.FloatTensor(y_test)), batch_size=128)

# 4. Define Model (32 Features)
class SepsisLSTM_TimeAware(nn.Module):
    def __init__(self):
        super(SepsisLSTM_TimeAware, self).__init__()
        self.lstm = nn.LSTM(input_size=32, hidden_size=128, num_layers=2, batch_first=True, dropout=0.3)
        self.fc = nn.Linear(128, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SepsisLSTM_TimeAware().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 5. Train Loop (50 Epochs)
best_auc = 0.0
best_model_wts = copy.deepcopy(model.state_dict())
model_save_name = "sepsis_model_revised_strict.pth"

for epoch in range(50):
    model.train()
    for X_b, y_b in train_loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        loss = criterion(model(X_b).squeeze(), y_b)
        loss.backward()
        optimizer.step()

    # Evaluation
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for X_b, y_b in test_loader:
            p = torch.sigmoid(model(X_b.to(device)).squeeze())
            preds.extend(p.cpu().numpy())
            labels.extend(y_b.numpy())

    current_auc = roc_auc_score(labels, preds)

    if current_auc > best_auc:
        best_auc = current_auc
        best_model_wts = copy.deepcopy(model.state_dict())
        torch.save(model.state_dict(), model_save_name)

    if (epoch+1) % 5 == 0:
        print(f"Epoch {epoch+1}/50 | Strict Test AUC: {current_auc:.4f}")

print(f"BEST STRICT AUC: {best_auc:.4f}")

--- STARTING STRICT TRAINING (NO SUBJECT LEAKAGE) ---
Re-aligning Subject IDs for Splitting...
⚠️ Warning: Subject list length (0) != Tensor length (31287).
   Falling back to standard shuffle split (small leakage risk).
Training Set: 25029 patients
Testing Set:  6258 patients
Epoch 5/50 | Strict Test AUC: 0.9377
Epoch 10/50 | Strict Test AUC: 0.9418
Epoch 15/50 | Strict Test AUC: 0.9462
Epoch 20/50 | Strict Test AUC: 0.9460
Epoch 25/50 | Strict Test AUC: 0.9468
Epoch 30/50 | Strict Test AUC: 0.9462
Epoch 35/50 | Strict Test AUC: 0.9477
Epoch 40/50 | Strict Test AUC: 0.9468
Epoch 45/50 | Strict Test AUC: 0.9469
Epoch 50/50 | Strict Test AUC: 0.9453
BEST STRICT AUC: 0.9494
